In [75]:
from transformers import AutoModelForCausalLM, AutoTokenizer

def extraer_props_atomicas(documento):
    
    model_id = "paascorb/ProposicionadorES-LFM2-1.2B"
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        torch_dtype="bfloat16",
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    
    input_ids = tokenizer.apply_chat_template(
        [{"role": "user", "content": documento}],
        add_generation_prompt=True,
        return_tensors="pt",
        tokenize=True,
    ).to(model.device)

    output = model.generate(
        input_ids,
        do_sample=True,
        temperature=0.3,
        min_p=0.15,
        repetition_penalty=1.05,
        max_new_tokens=2048,
    )

    return tokenizer.decode(output[0], skip_special_tokens=False).split("<|im_start|>assistant")[1].strip("<|im_end|>")

AttributeError: 'str' object has no attribute 'content'

In [ ]:
import wikipedia
from pylate import indexes, models, retrieve
import pandas as pd

animales = ["Balaenoptera musculus", "Corvus corax", "Amphiprion ocellaris", "Chelonia mydas", "Coccinellidae", "Agalychnis callidryas"]
literatura = ["Colette", "Émile Zola", "George Sand", "Ramón María del Valle-Inclán", "Miguel de Cervantes" ,"Rosalía de Castro"]

wikipedia.set_lang('es')

model = models.ColBERT(
    model_name_or_path="LiquidAI/LFM2-ColBERT-350M",
)
model.tokenizer.pad_token = model.tokenizer.eos_token

index = indexes.PLAID(
    index_folder="uqap_index",
    index_name="uqap",
    override=True,
)

base_datos = []
documents = []
cont = 1
documents_ids = []
for elem in animales+literatura:
    page = wikipedia.page(elem)
    for sect in page.sections:
        proposiciones = list(filter(None, extraer_props_atomicas(page.section(sect)).splitlines()))
        for prop in proposiciones:
            documents.append(prop)
            documents_ids.append(str(cont))
            base_datos.append({"id": cont, "título": page.title, "seccion": sect, "texto": prop})
            cont += 1

documents_embeddings = model.encode(
    documents,
    batch_size=32,
    is_query=False,
    show_progress_bar=True,
)

index.add_documents(
    documents_ids=documents_ids,
    documents_embeddings=documents_embeddings,
)
df = pd.DataFrame(base_datos)
df.to_csv("UQAP/base_datos.csv", index=False)

In [ ]:
from pylate import indexes, models, retrieve
import pandas as pd

prueba_pregunta = "¿A qué familia pertenece la especie Balaenoptera musculus?"

df = pd.read_csv("UQAP/base_datos.csv")

index = indexes.PLAID(
    index_folder="uqap_index",
    index_name="uqap",
)

retriever = retrieve.ColBERT(index=index)

queries_embeddings = model.encode(
    [prueba_pregunta],
    batch_size=32,
    is_query=True,
    show_progress_bar=True,
)

scores = retriever.retrieve(
    queries_embeddings=queries_embeddings,
    k=10,
)

df.loc[df['id'].isin([int(x["id"]) for x in scores[0]])]

In [ ]:
from pylate import indexes, models, retrieve
import pandas as pd

def recuperar_documento(pregunta):
    
    df = pd.read_csv("UQAP/base_datos.csv")
    
    index = indexes.PLAID(
        index_folder="uqap_index",
        index_name="uqap",
    )
    
    retriever = retrieve.ColBERT(index=index)
    
    queries_embeddings = model.encode(
        [prueba_pregunta],
        batch_size=32,
        is_query=True,
        show_progress_bar=True,
    )
    
    scores = retriever.retrieve(
        queries_embeddings=queries_embeddings,
        k=10,
    )
    
    return df.loc[df['id'].isin([int(x["id"]) for x in scores[0]])]